# 🔌 Database Connection — Playground

Notebook untuk mencoba-coba koneksi Python ↔ PostgreSQL dan melakukan query.

---

## Daftar Isi
1. **Setup** — Import & koneksi
2. **Cara 1: psycopg2** — Query low-level (raw SQL)
3. **Cara 2: SQLAlchemy + Pandas** — Query high-level (langsung DataFrame)
4. **Explorasi Silver** — Lihat 12 tabel silver
5. **Explorasi Gold** — Lihat OBT 124 kolom
6. **Query Lanjutan** — Contoh analisis dari database

---
## 1. Setup

In [ ]:
import sys
from pathlib import Path

# Tambahkan root project ke path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import psycopg2

# Import helper functions dari project kita
from config.config import DB_CONFIG, DB_URL
from src.utils.helpers import get_connection, get_engine

print("✅ Import berhasil")
print(f"DB Config: {DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['dbname']} (user: {DB_CONFIG['user']})")

---
## 2. Cara 1: psycopg2 (Low-Level)

Psycopg2 adalah driver Python untuk PostgreSQL. Kita kirim SQL mentah dan dapat hasilnya sebagai **tuple**.

### 2.1 Buka Koneksi & Cek Info Database

In [ ]:
# Buka koneksi
conn = get_connection()
cur = conn.cursor()  # cursor = "pointer" untuk mengirim query

# Query info database
cur.execute("SELECT current_database(), current_user, version()")
db, user, version = cur.fetchone()  # ambil 1 baris hasil

print(f"📦 Database : {db}")
print(f"👤 User     : {user}")
print(f"🐘 Version  : {version[:60]}...")

### 2.2 Lihat Semua Schema yang Ada

In [ ]:
cur.execute("""
    SELECT schema_name 
    FROM information_schema.schemata 
    WHERE schema_name NOT LIKE 'pg_%' 
      AND schema_name != 'information_schema'
    ORDER BY schema_name
""")

schemas = cur.fetchall()  # ambil semua baris
print("📂 Schemas:")
for (schema,) in schemas:
    print(f"   └── {schema}")

### 2.3 Lihat Semua Tabel per Schema

In [ ]:
cur.execute("""
    SELECT table_schema, table_name 
    FROM information_schema.tables 
    WHERE table_schema IN ('silver', 'gold')
    ORDER BY table_schema, table_name
""")

tables = cur.fetchall()
print(f"📋 Total tabel: {len(tables)}\n")

current_schema = None
for schema, table in tables:
    if schema != current_schema:
        print(f"\n🗂️  {schema}")
        current_schema = schema
    print(f"   ├── {table}")

### 2.4 Query Sederhana dengan psycopg2

In [ ]:
# Query: 5 tim teratas berdasarkan goals
cur.execute("""
    SELECT club, goals, xg, conversion_pct 
    FROM silver.teams_attacking_overall 
    ORDER BY goals DESC 
    LIMIT 5
""")

# Ambil nama kolom dari cursor
columns = [desc[0] for desc in cur.description]
rows = cur.fetchall()

print(f"Kolom: {columns}\n")
for row in rows:
    print(f"  {row[0]:20s} | goals={row[1]:3d} | xg={row[2]:6.2f} | conv={row[3]:.2f}%")

In [ ]:
# Jangan lupa tutup koneksi psycopg2 setelah selesai!
conn.close()
print("🔒 Koneksi psycopg2 ditutup")

---
## 3. Cara 2: SQLAlchemy + Pandas (High-Level)

Cara yang **lebih praktis** — hasil query langsung jadi DataFrame!

### 3.1 Buat Engine

In [ ]:
engine = get_engine()
print(f"✅ Engine: {engine.url}")

### 3.2 Query → DataFrame (1 baris kode!)

In [ ]:
# Langsung jadi DataFrame!
df = pd.read_sql("SELECT * FROM silver.teams_attacking_overall ORDER BY goals DESC", engine)
df.head()

### 3.3 Perbandingan: psycopg2 vs SQLAlchemy

| Aspek | psycopg2 | SQLAlchemy + Pandas |
|---|---|---|
| Hasil | List of tuples | DataFrame |
| Kode | ~5 baris | 1 baris |
| Kapan pakai | DDL, CALL, INSERT | SELECT (analisis) |
| Contoh | `cur.execute(sql)` | `pd.read_sql(sql, engine)` |

---
## 4. Explorasi Silver (12 Tabel)

In [ ]:
# Lihat semua tabel silver dan jumlah kolom + rows-nya
df_tables = pd.read_sql("""
    SELECT table_name,
           (SELECT COUNT(*) FROM information_schema.columns c 
            WHERE c.table_schema = t.table_schema AND c.table_name = t.table_name) AS cols
    FROM information_schema.tables t
    WHERE t.table_schema = 'silver'
    ORDER BY table_name
""", engine)

print(f"📊 Silver tables: {len(df_tables)} tabel\n")
df_tables

In [ ]:
# Contoh: lihat tabel attacking misc (ada fast_breaks!)
df_att_misc = pd.read_sql("SELECT * FROM silver.teams_attacking_misc ORDER BY fast_breaks_total DESC", engine)
df_att_misc

In [ ]:
# Contoh: lihat tabel defending overall (gol yang kebobolan)
df_def = pd.read_sql("SELECT * FROM silver.teams_defending_overall ORDER BY goals ASC", engine)
df_def.head()

---
## 5. Explorasi Gold (OBT — 124 Kolom)

In [ ]:
# Load seluruh gold table
df_gold = pd.read_sql("SELECT * FROM gold.teams_statistics", engine)
print(f"📊 Gold OBT: {df_gold.shape[0]} rows × {df_gold.shape[1]} kolom")
print(f"\n📋 Daftar kolom:")
for i, col in enumerate(df_gold.columns, 1):
    print(f"  {i:3d}. {col}")

In [ ]:
# Preview beberapa kolom penting
df_gold[[
    'club', 'att_goals', 'att_xg', 'att_fast_breaks_total', 'att_fast_breaks_goals',
    'def_goals_conceded', 'def_avg_possession_pct', 'prs_ppda'
]].sort_values('att_goals', ascending=False)

---
## 6. Query Lanjutan — Contoh Analisis dari Database

Anda bisa langsung menulis query SQL yang kompleks, hasilnya langsung jadi DataFrame.

In [ ]:
# Contoh 1: JOIN langsung di SQL (dari silver, tanpa perlu gold!)
df_join = pd.read_sql("""
    SELECT 
        a.club,
        a.goals AS att_goals,
        do2.goals AS def_goals_conceded,
        (a.goals - do2.goals) AS goal_difference,
        dda.avg_possession_pct,
        am.fast_breaks_total,
        am.fast_breaks_goals
    FROM silver.teams_attacking_overall a
    JOIN silver.teams_defending_overall do2 ON do2.club = a.club
    JOIN silver.teams_defending_defensive_action dda ON dda.club = a.club
    JOIN silver.teams_attacking_misc am ON am.club = a.club
    ORDER BY goal_difference DESC
""", engine)

print("🏆 Goal Difference + Possession + Fast Breaks\n")
df_join

In [ ]:
# Contoh 2: Query langsung dari Gold (1 tabel, tanpa JOIN)
df_style = pd.read_sql("""
    SELECT 
        club,
        def_avg_possession_pct AS possession,
        att_fast_breaks_per_game AS fast_breaks_pg,
        prs_ppda AS ppda,
        seq_buildups_per_game AS buildups_pg,
        seq_direct_attacks_per_game AS direct_attacks_pg,
        def_goals_conceded_per_game AS goals_conceded_pg
    FROM gold.teams_statistics
    ORDER BY possession DESC
""", engine)

print("⚽ Style Indicators dari Gold OBT\n")
df_style

In [ ]:
# Contoh 3: Aggregasi di SQL
df_agg = pd.read_sql("""
    SELECT 
        'LaLiga 2024/25' AS league,
        AVG(att_goals_per_game) AS avg_goals_per_game,
        AVG(def_avg_possession_pct) AS avg_possession,
        AVG(prs_ppda) AS avg_ppda,
        AVG(att_fast_breaks_per_game) AS avg_fast_breaks_per_game,
        SUM(att_goals) AS total_goals,
        SUM(att_fast_breaks_goals) AS total_fast_break_goals
    FROM gold.teams_statistics
""", engine)

print("📊 Liga Overview\n")
df_agg.T  # transpose supaya mudah dibaca

---
## 🧹 Cleanup

In [ ]:
# Tutup engine SQLAlchemy
engine.dispose()
print("🔒 Engine disposed. Koneksi ditutup.")

---
## 📝 Cheatsheet

```python
# === SETUP (sekali di awal notebook) ===
from src.utils.helpers import get_engine
engine = get_engine()

# === QUERY (berulang kali) ===
df = pd.read_sql("SELECT * FROM gold.teams_statistics", engine)
df = pd.read_sql("SELECT club, att_goals FROM silver.teams_attacking_overall WHERE goals > 50", engine)

# === CLEANUP (di akhir notebook) ===
engine.dispose()
```